Part A: Setting up the directory, packages, and environment for the datasets



In [ ]:
# Create the Google Drive connection (only for Google Drive)
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Import the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from scipy.stats import zscore

Part B: Downloading the datasets

In [ ]:
# In a different IDE, just adjust the file path

# Import holidays dataset
df_holidays = pd.read_csv('/content/drive/MyDrive/ProjectData/holidays.csv')
display(df_holidays.head())

In [ ]:
# Import stores dataset
df_stores = pd.read_csv('/content/drive/MyDrive/ProjectData/stores.csv')
display(df_stores.head())

In [ ]:
# Import oil dataset
df_oil = pd.read_csv('/content/drive/MyDrive/ProjectData/oil.csv')
display(df_oil.head())

In [ ]:
# Import time series dataset
df_timeseries = pd.read_csv('/content/drive/MyDrive/ProjectData/timeseries.csv')
display(df_timeseries.head())

Part C: Exploratory Data Analysis

In [ ]:
print("df_holidays head:")
display(df_holidays.head())

print("\nVisualizing df_holidays data:")
# Bar plot for holiday types
plt.figure(figsize=(10, 6))
df_holidays['locale'].value_counts().plot(kind='bar')
plt.title('Distribution of Holiday Locales')
plt.xlabel('Holiday Locale')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
print("df_stores head:")
display(df_stores.head())

# Bar plot for store clusters
plt.figure(figsize=(10, 5))
df_stores['region'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution of Regions of Store Locations')
plt.xlabel('Region')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Oil dataset - Convert date format and plotted the line graph
df_oil['date'] = pd.to_datetime(df_oil['date'])
df_oil.plot(x='date', y='dcoilwtico')
plt.show()

In [ ]:
# Oil dataset
# Impute missing values in the 'dcoilwtico' column of df_oil using linear interpolation
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].interpolate(method='linear')

# Display the head of the DataFrame to show the changes and check for remaining NaNs
print("DataFrame head after interpolation:")
display(df_oil.head())

# Check if there are any remaining NaNs in the 'dcoilwtico' column
print("Number of missing values in 'dcoilwtico' after interpolation:", df_oil['dcoilwtico'].isnull().sum())

In [ ]:
# Oil dataset - Let's check it again to see if the values were filled
df_oil['date'] = pd.to_datetime(df_oil['date'])
df_oil.plot(x='date', y='dcoilwtico')
plt.show()

In [ ]:
# Time Series dataset - Convert date format and plotted the line graph
df_timeseries['date'] = pd.to_datetime(df_timeseries['date'])
df_timeseries.plot(x='date', y='unit_sales')
plt.show()

In [ ]:
# Time Series dataset - Use ADF test function to check if it is stationarity (H0 = non-stationary vs. H1 = stationary)
def run_adf(timeseries):
    """Performs Augmented Dickey-Fuller test on a given time series."""
    adf_result = adfuller(timeseries)
    p_value = adf_result[1]
    print(f'P-value is {p_value}')
    if p_value <= 0.05:
        print("The series is stationary (reject H0).")
    else:
        print("The series is non-stationary (fail to reject H0).")

In [ ]:
# Time Series dataset - Test stationarity on original series
run_adf(df_timeseries['unit_sales'])

In [ ]:
# Time Series dataset - Decompose the time series into Trend × Seasonality × Residual (multiplicative model)
df_timeseries_indexed = df_timeseries.set_index('date')
decomp_mul = seasonal_decompose(df_timeseries_indexed['unit_sales'], model="multiplicative", period=7)
decomp_mul.plot()
plt.suptitle("Demonstration", y=1.02)
plt.show()

In [ ]:
# Time Series dataset - Detecting Outliers with a Boxplot
df_timeseries.plot.box(y='unit_sales', title='Boxplot of Unit Sales')
plt.ylabel('Unit Sales')
plt.show()

In [ ]:
# Time Series dataset -Calculate Z-scores for the 'unit_sales' column to provide a statistical measure for outliers
df_timeseries['unit_sales_zscore'] = zscore(df_timeseries['unit_sales'])

# Display the head of the DataFrame with the new Z-score column
print("DataFrame head with Z-scores:")
display(df_timeseries.head())

In [ ]:
# Time Series dataset - Create an autocorrelation plot for 'unit_sales'
plt.figure(figsize=(12, 6))
plot_acf(df_timeseries['unit_sales'], lags=30)
plt.title('Autocorrelation Function for Unit Sales')
plt.xlabel('Lags')
plt.ylabel('Autocorrelation')
plt.show()

Part D: Export a .csv file (optional)

In [ ]:
# Export a .csv file
df_timeseries.to_csv('output.csv', index=False)
print('df_timeseries exported to output.csv')